# Sonic 2 — Dreamcast autobuilder

Builds a bootable `Sonic2.cdi` for the SEGA Dreamcast from the Sonic 2
Dreamcast port.

**You supply one file, and you upload it yourself.** This notebook ships no
game data — no `Data.rsdk`, no music, no sound effects. It is a build script;
the game is yours.

| | what |
|---|---|
| `Data.rsdk` | from your own copy of **Sonic the Hedgehog 2 (2013), mobile remaster** |
| `settings.ini` | *optional* — your own, if you want it on the disc. Without one the engine uses its built-in defaults |

Drag them into `/content` using the **Files** tab in the left sidebar (the folder
icon). It lives in this runtime, is used, and is gone when the session ends —
**nothing is written to your Google Drive, and the notebook never asks to mount
it.** The compiler and the build configuration are downloaded from the project's
own account; that is the only Drive involved.

### One upload, like the Sonic 1 and Sonic CD builders

The notebook pulls the music, the global sound effects and the 2013 menu art out
of `Data.rsdk` itself and converts them. The datafile then ships to the disc
untouched, which is where everything else — sprites, tilesets, scripts, stage
sfx — comes from.

RSDKv4 makes that harder than RSDKv3 did. A v3 pack stores plaintext paths, so
Sonic CD's builder can simply walk its directory table. **v4 stores only an MD5
of each path**, so there is no list to read back — every name has to be known
before it can be looked up. `dreamcast/rsdkv4_extract.py` gets around it by
bootstrapping: `Data/Game/GameConfig.bin` is at a path we know and names every
sound effect, and the music and menu names come from
`dreamcast/rsdkv4_filelist.txt` bundled beside the script.

### One difference from the Sonic 1 builder worth knowing

Sonic 1's global sound effects are `.wav`. **Sonic 2's are `.ogg`.** Both hold
the same headerless signed 8-bit mono PCM once converted, and both keep their
original name because GameConfig resolves assets by filename — but a pipeline
that went looking for `*.wav` here would convert nothing at all and say so only
in the verify step. `dreamcast/verify_assets.sh` checks for both failure modes.

---

## Two sections

**Section A** compiles the ELF and converts your assets into a *build kit*.

**Section B** masters that kit into a CDI, in about two minutes.

They're split because the kit is stable and the disc isn't: change
`settings.ini`, rename the disc, flip a mkdcdisc flag, and you re-run **B
alone** without recompiling.

**Run both in the same session.** The kit lives in the runtime, so it goes when
the session does.

## Before you press anything

- **Leave this tab open.** Colab reclaims idle runtimes after about 90 minutes,
  and a reclaimed runtime loses everything.
- Runtime → Change runtime type → **CPU**. A GPU does nothing here and only
  makes a runtime harder to get.
- Expect roughly 10 minutes end to end **if `TOOLCHAIN_GDRIVE_ID` is set**.
  Measured on a 2-core Linux box with the toolchain already unpacked: the ELF
  compiled in 15 seconds and asset conversion took about 2 minutes, most of it
  VQ-compressing the menu textures. Colab's CPUs are slower and the toolchain
  download comes on top, hence the margin. Without a prebuilt toolchain, add
  60–90 minutes to build GCC from source — see the note at the top of Section A.
- `REPO_URL` is **empty** and has to be filled in. See the note in Section A.

### A note on the KallistiOS version

The port is built against **KOS master**, which reports itself as 2.3.0. That is
a version string, not a release — there is no `v2.3.0` tag, and the newest tag is
`v2.2.2`. Section A therefore pins an exact commit (`KOS_COMMIT`) rather than
following a branch.

The commit pinned here is the same one the Sonic 1 builder uses, and it is not a
guess: a Sonic 2 boot log from the development machine reads

```
KallistiOS v2.3.0 [dreamcast/pristine]
  Git revision: 14bc62b6-dirty
  sh-elf-gcc (GCC) 14.4.1 20260715
```

so both ports are developed against the same KOS tree. On master the toolchain
builder lives at `utils/kos-chain`, not `utils/dc-chain`.


In [ ]:
#@title ▶ Section A — build the ELF and convert your assets { display-mode: "form" }
#
# Press play. Roughly 10 minutes with a prebuilt toolchain; leave the tab open.
#
# Nothing here is configurable on purpose: the toolchain, the KOS version
# and the build flags are the combination this port is tested with, and a
# half-changed one produces a disc that fails in ways nobody can help with.
# The code is under "Show code" in the cell menu if you want to read it.
#
# Nothing here touches your Google Drive. Your Data.rsdk is uploaded into this
# runtime, used, and gone when the session ends.

# ── The port ──────────────────────────────────────────────────────────
#
# DELIBERATELY EMPTY. The Sonic 2 port's repository URL is not written in here
# because guessing one would be worse than asking: a plausible-looking wrong URL
# clones somebody else's tree and compiles it, and the failure arrives twenty
# minutes later as a cmake error nobody can place.
#
# Put the port's git URL here. A local path works too -- git clone accepts one --
# so if the source is already in this runtime, point at that.
REPO_URL = ""
REPO_REF = "main"                    # branch or tag

# ── Toolchain ─────────────────────────────────────────────────────────
# A prebuilt /opt/toolchains/dc, built on Ubuntu 22.04 so its binaries run on
# Colab's glibc, and AT the path /opt/toolchains/dc because KOS bakes absolute
# paths into environ.sh.
#
# DELIBERATELY EMPTY, and read the next paragraph before filling it in, because
# the obvious archive is NOT the combination this port is developed against.
#
# THE SONIC 1 AUTOBUILDER'S ARCHIVE, INSPECTED 2026-11-21 (136 MiB xz, 434 MiB
# unpacked, relative paths under opt/toolchains/dc, so it unpacks correctly):
#
#     KOS      branch v2.2.x at 5c274f02, i.e. v2.2.2-1-g5c274f02, working tree
#              clean, include/kos/version.h reports 2.2.2, builder is
#              utils/dc-chain
#     GCC      14.3.0        newlib 4.5.0        no arm-eabi
#
# The Sonic 2 port is developed against KOS MASTER at 14bc62b6 (reports 2.3.0,
# builder utils/kos-chain) with sh-elf-gcc 14.4.1. So that archive is a
# different KOS line and a different GCC, not the same combination.
#
# It will very probably still build. All 13 KOS headers and all 53 KOS symbols
# this port references exist in 2.2.2; of the five declarations that differ,
# four are cosmetic (uint32/uint32_t, size_t/uint32_t for a 32-bit count,
# function vs static inline, int vs unsigned int) and the fifth -- fs_total
# returning size_t on 2.2.2 and ssize_t on master -- is already cast explicitly
# in DCAudio.cpp with a comment anticipating exactly that drift. Nothing in the
# port is #if'd on KOS_VERSION. But that is a header-surface argument, not a
# compile, so treat the first run with this archive as the test of it.
#
# NOTE THAT KOS_COMMIT BELOW IS INERT ON THIS PATH. With an archive supplied,
# have_compiler() and have_kos() both succeed, fetch_kos() never runs, and the
# pin is never checked against anything. The KOS identity report further down
# is what actually tells you which KOS the ELF got built against.
#
# To build at the pinned commit instead without paying for a new toolchain: the
# archive's KOS is a FULL clone and already contains 14bc62b6, so checking it
# out in place and re-running `make -C /opt/toolchains/dc/kos` rebuilds KOS
# master with GCC 14.3.0 in 10-15 minutes rather than 60-90. Untested here.
#
# Left blank entirely, the cell builds the compiler from source below, which
# costs 60-90 minutes.
#
# TO MAKE THIS FAST, ONCE, FOR EVERYONE:
#   1. Run this cell as-is and let kos-chain build.
#   2. tar -C / -cf dc-toolchain.tar --zstd opt/toolchains/dc
#   3. Upload that to Drive, share it "anyone with the link", and paste the
#      file id below. Every later run then takes ~10 minutes instead of ~100.
TOOLCHAIN_GDRIVE_ID = ""
TOOLCHAIN_URL       = ""

# KOS's environ.sh. Blank uses KOS's stock sample config, which is correct for
# this port; set an id only if the project later ships a tuned one.
ENVIRON_GDRIVE_ID = ""

# ── KallistiOS ────────────────────────────────────────────────────────
# A COMMIT, not a tag, and deliberately so. There is no v2.3.0 tag to clone:
# 2.3.0 is the version master reports in include/kos/version.h, while the newest
# release tag is v2.2.2.
#
# This is the same commit the Sonic 1 builder pins, taken from a Sonic 2 boot
# log off the development machine:
#     KallistiOS v2.3.0 [dreamcast/pristine]
#       Git revision: 14bc62b6-dirty
#       sh-elf-gcc (GCC) 14.4.1 20260715
# Both ports are developed against the same KOS tree, so tracking the branch
# instead would mean every run built against a KOS nobody had tested.
KOS_COMMIT = "14bc62b675ce41e2e05f34936f5e6b6a5313bd3a"   # master, 2026-07-11
KOS_LABEL  = "master @ 14bc62b6 (reports 2.3.0)"

# A kos-chain profile name, i.e. a file in utils/kos-chain/profiles/dreamcast/.
# The development machine runs sh-elf-gcc 14.4.1, which is the "14.4.1-dev"
# profile: GCC 14 built from git, so it is slower and moves underneath you.
# "14.4.0" is the same series from a fixed release tarball, which is what an
# unattended build wants. Set it to "14.4.1-dev" to match the dev machine
# exactly.
TOOLCHAIN_PROFILE = "14.4.0"

EXTRA_CMAKE_FLAGS = ""   # the shipping configuration is the default

# ── Where work happens (all inside this runtime) ──────────────────────
WORK      = "/content/work"
KIT_DIR   = f"{WORK}/build-kit"
CACHE_DIR = f"{WORK}/cache"

import os, sys, time, subprocess, shutil, textwrap, collections

_t0 = {}
def step(name):
    print(f"\n\033[1m▶ {name}\033[0m", flush=True)
    _t0[name] = time.time()
    return name

def done(name):
    d = time.time() - _t0[name]
    print(f"\033[32m✔ {name} — {int(d // 60)}m {int(d % 60):02d}s\033[0m", flush=True)

def sh(cmd, cwd=None, env=None, quiet=False):
    """Run a shell command, streaming its output. Raises on non-zero exit."""
    e = dict(os.environ); e.update(env or {})
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    # A BOUNDED deque, not a list: holding every line a command ever printed is
    # harmless for a 20-line apt run and ruinous for a compile that emits
    # hundreds of thousands while GCC already has most of the RAM.
    tail = collections.deque(maxlen=300)
    for line in p.stdout:
        tail.append(line)
        if not quiet:
            sys.stdout.write(line)
    p.wait()
    if p.returncode:
        if quiet:
            print("".join(tail))
        raise SystemExit(f"command failed (exit {p.returncode}): {cmd}")
    return "".join(tail)

os.makedirs(WORK, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

if not REPO_URL.strip() or REPO_URL.startswith("<<"):
    raise SystemExit(
        "REPO_URL is empty.\n"
        "Open this cell (Show code in the cell menu) and put the port's git URL\n"
        "in REPO_URL near the top.\n\n"
        "If you already have the source in this runtime, point REPO_URL at it\n"
        "as a local path instead - git clone accepts one.")

print("\033[1mSonic 2 — Dreamcast autobuilder\033[0m")
print(f"  pin    KOS {KOS_LABEL} + GCC {TOOLCHAIN_PROFILE}  (from-source path; step 3 reports what is actually used)")
print(f"  repo   {REPO_URL} @ {REPO_REF}")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  1/6  your Data.rsdk\n══════════════════════════════════════════════════════════════\033[0m")
#
# One upload:
#     Data.rsdk   from your own copy of Sonic 2 (2013), mobile remaster
#
# The music, the global sound effects and the 2013 menu art are extracted from
# it in step 5; the same file then goes on the disc untouched.
#
# Picked up from /content by extension rather than by exact name, so a browser
# that appended " (1)" does not send anybody back to the sidebar.

def _pick(exts, label, min_bytes=0):
    hits = []
    for d in ("/content", WORK):
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            p = os.path.join(d, f)
            if os.path.isfile(p) and f.lower().endswith(exts) and os.path.getsize(p) >= min_bytes:
                hits.append(p)
    if not hits:
        raise SystemExit(
            f"No {label} found in /content.\n"
            f"Open the Files tab in the left sidebar (folder icon) and drag it in,\n"
            f"then press play on this cell again.")
    if len(hits) > 1:
        print(f"  several candidates for {label}; using the largest")
    return max(hits, key=os.path.getsize)

n = step("locate your upload")
rsdk_path = _pick((".rsdk",), "Data.rsdk", min_bytes=1 << 20)
print(f"  Data.rsdk   {rsdk_path}   {os.path.getsize(rsdk_path) / 1e6:.1f} MB")

# Optional second upload. Matched by extension like Data.rsdk, so "settings
# (1).ini" from a browser still counts; it goes on the disc as settings.ini.
_inis = sorted(os.path.join("/content", f) for f in os.listdir("/content")
               if f.lower().endswith(".ini") and os.path.isfile(os.path.join("/content", f)))
ini_upload = next((p for p in _inis if os.path.basename(p).lower() == "settings.ini"),
                  _inis[0] if _inis else None)
print(f"  settings    {ini_upload or 'none uploaded (optional) — engine defaults'}")

# The wrong file here would not surface until step 5, on the far side of the
# whole compile. Check the header now. RSDKv4 packs start "RSDKvB"; a Sonic CD
# datafile is RSDKv3 and belongs to the other builder.
with open(rsdk_path, "rb") as f:
    _magic = f.read(6)
if _magic != b"RSDKvB":
    raise SystemExit(
        f"{rsdk_path} does not look like an RSDKv4 datafile — it starts\n"
        f"{_magic!r}, not b'RSDKvB'.\n\n"
        "Sonic 1 and Sonic 2 (2013) are RSDKv4. Sonic CD is RSDKv3 and needs\n"
        "the Sonic CD builder instead.")
# Sonic 1's pack and Sonic 2's are both RSDKvB, so the magic cannot tell them
# apart. A size check can hint: Sonic 2's retail pack is around 47 MB, Sonic 1's
# is appreciably smaller. Warn rather than fail -- a modified pack is legitimate.
_mb = os.path.getsize(rsdk_path) / 1e6
if _mb < 20:
    print(f"  \033[33mnote: {_mb:.0f} MB is small for a Sonic 2 pack (~47 MB retail).")
    print("  If this is the Sonic 1 datafile, the build will finish and the game")
    print("  will not. Nothing downstream can tell the two apart.\033[0m")
done(n)

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  2/6  system packages and the port's source\n══════════════════════════════════════════════════════════════\033[0m")

n = step("apt packages")
sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "build-essential libjpeg-dev libpng-dev libelf-dev texinfo libgmp-dev libmpfr-dev "
   "libmpc-dev libisl-dev zstd bison flex gettext "
   "cmake ffmpeg git wget curl patch bzip2 gawk python3 python3-pil python3-numpy > /dev/null",
   quiet=True)
done(n)

SRC = f"{WORK}/RSDKv4-Decompilation"
if not os.path.isdir(SRC):
    n = step("clone the port")
    sh(f"git clone --recursive --depth 1 --branch {REPO_REF} {REPO_URL} {SRC}")
    done(n)

# The asset scripts must be executable; a zip download or a checkout on a
# filesystem that drops the bit leaves them without it. Cheap to make certain.
sh(f"chmod +x {SRC}/dreamcast/*.sh {SRC}/dreamcast/*.py 2>/dev/null || true")

# Preflight the asset pipeline NOW, not in step 5 on the far side of the
# compile. Two files the kit needs and cannot synthesise:
#
#   rsdkv4_extract.py    game-agnostic; shared with the Sonic 1 kit
#   rsdkv4_filelist.txt  per-game, and the thing an MD5-keyed pack cannot give
#                        back. dreamcast/make_filelist.py builds it.
_missing = [f for f in ("rsdkv4_extract.py", "rsdkv4_filelist.txt",
                        "generate_assets.sh", "verify_assets.sh")
            if not os.path.isfile(f"{SRC}/dreamcast/{f}")]
if _missing:
    raise SystemExit(
        "The port's dreamcast/ folder is missing: " + ", ".join(_missing) + "\n\n"
        "rsdkv4_extract.py is game-agnostic — copy it from the Sonic 1 kit.\n"
        "rsdkv4_filelist.txt is per-game and has to be generated, because an\n"
        "RSDKv4 pack stores only an MD5 of each path and cannot be enumerated:\n\n"
        "    cd dreamcast\n"
        "    ./make_filelist.py --data <extracted Data> --engine ../RSDKv4 \\\n"
        "                       --out rsdkv4_filelist.txt\n\n"
        "Commit both, then re-run this cell.")

# And the texture converter, which lives OUTSIDE dreamcast/. menu_step_1_dtex.sh
# looks for ../tools/dtex_convert.py first and dreamcast/dtex_convert.py second.
# Checked here because the first full run of this notebook against the real
# working tree compiled the ELF and THEN died at the menu step for want of it --
# and in that tree tools/dtex_convert.py is untracked, so a fresh clone would
# not have it either.
if not any(os.path.isfile(p) for p in (f"{SRC}/tools/dtex_convert.py",
                                         f"{SRC}/dreamcast/dtex_convert.py")):
    raise SystemExit(
        "dtex_convert.py is missing — neither tools/dtex_convert.py nor\n"
        "dreamcast/dtex_convert.py exists in the clone.\n\n"
        "The 2013 menu textures cannot be converted without it, and the menu is\n"
        "mostly invisible on hardware with PNGs. If it exists in your working\n"
        "tree, check it is committed:  git add tools/dtex_convert.py")

for tool in ("ffmpeg", "cmake", "git"):
    if not shutil.which(tool):
        raise SystemExit(f"{tool} is missing after apt — cannot continue.")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  3/6  the Dreamcast toolchain\n══════════════════════════════════════════════════════════════\033[0m")

DC = "/opt/toolchains/dc"

def have_compiler():
    cc = f"{DC}/sh-elf/bin/sh-elf-gcc"
    return os.path.isfile(cc) and subprocess.run([cc, "--version"],
                                                 capture_output=True).returncode == 0

def have_kos():
    return os.path.isfile(f"{DC}/kos/lib/dreamcast/libkallisti.a")

def fetch_kos():
    """Put the pinned KOS commit at {DC}/kos.

    git cannot `clone --branch` a commit SHA, so this is init + fetch, which
    GitHub does allow for an arbitrary commit and which stays shallow. The
    fallback is a full clone, for a mirror that refuses a bare SHA.
    """
    if os.path.isdir(f"{DC}/kos/.git"):
        return
    repo = "https://github.com/KallistiOS/KallistiOS.git"
    sh(f"mkdir -p {DC}/kos")
    try:
        sh(f"cd {DC}/kos && git init -q . && git remote add origin {repo} && "
           f"git fetch --depth 1 origin {KOS_COMMIT} && "
           f"git -c advice.detachedHead=false checkout -q FETCH_HEAD", quiet=True)
    except SystemExit:
        print("  shallow fetch by commit refused; falling back to a full clone")
        sh(f"rm -rf {DC}/kos && git clone -q {repo} {DC}/kos")
        sh(f"git -C {DC}/kos -c advice.detachedHead=false checkout -q {KOS_COMMIT}")
    head = sh(f"git -c safe.directory='*' -C {DC}/kos rev-parse HEAD", quiet=True).strip()
    if head != KOS_COMMIT:
        raise SystemExit(f"KOS is at {head}, expected {KOS_COMMIT}")

if have_compiler():
    print("  compiler already present")
elif TOOLCHAIN_URL or TOOLCHAIN_GDRIVE_ID:
    n = step("download the prebuilt toolchain")
    tar = "/tmp/dc-toolchain.tar"
    if TOOLCHAIN_GDRIVE_ID:
        # Google Drive's large-file interstitial defeats plain wget; gdown
        # handles the confirmation token.
        sh("pip install -q gdown")
        sh(f"gdown -O {tar} {TOOLCHAIN_GDRIVE_ID}")
    else:
        sh(f"wget -q --show-progress -O {tar} '{TOOLCHAIN_URL}'")

    # CHECK WHAT ACTUALLY LANDED, before handing it to tar.
    #
    # Two ways this goes wrong, and tar's complaint points at the wrong thing in
    # both. gdown returns Google's HTML permission page -- with a 200 -- when a
    # Drive file is not really shared "anyone with the link". And a file-host
    # PAGE url is not a direct download: a Gofile /d/<id> folder or a Drive
    # /file/d/<id>/view link saves the JavaScript shell instead, because the
    # real link is resolved client-side. Either way this is a few KB of HTML and
    # tar says "does not look like a tar archive", which reads as a corrupt
    # download rather than a sharing or url problem.
    with open(tar, "rb") as f:
        head = f.read(512)
    sz = os.path.getsize(tar)
    kind = None
    for magic, name in ((b"\x28\xb5\x2f\xfd", "zstd"), (b"\x1f\x8b", "gzip"),
                        (b"\xfd7zXZ\x00", "xz"), (b"BZh", "bzip2")):
        if head.startswith(magic):
            kind = name
            break
    if kind is None and head[257:262] == b"ustar":
        kind = "tar"
    if kind is None:
        stripped = head.lstrip()
        html = stripped[:1] == b"<" or b"<html" in head.lower()
        raise SystemExit(
            f"{tar} is {sz / 1e6:.2f} MB and is not an archive"
            + (" — it is an HTML page.\n\n" if html else ".\n\n")
            + ("A folder or viewer PAGE url is not a direct download. A Gofile\n"
               "/d/<id> link and a Drive /file/d/<id>/view link both resolve the\n"
               "real file client-side, so wget saves the page.\n\n"
               "Use TOOLCHAIN_GDRIVE_ID with just the Drive FILE ID — the long\n"
               "string between /d/ and /view in the share link — and gdown will\n"
               "handle the interstitial. If that is what you did, the file is\n"
               "probably not shared with 'anyone with the link', which is what\n"
               "returns the permission page.\n"
               if html else
               "Expected zstd, gzip, xz, bzip2 or plain tar. Check the url or\n"
               "file id points at the toolchain archive itself.\n"))
    print(f"  {sz / 1e6:.0f} MB, {kind}")
    sh(f"tar --auto-compress -xf {tar} -C / || tar -xf {tar} -C /")
    os.remove(tar)
    if not have_compiler():
        raise SystemExit("The archive unpacked but sh-elf-gcc is not at "
                         f"{DC}/sh-elf/bin — was it built for that exact path?")
    done(n)
else:
    print("\n\033[33mNo prebuilt toolchain configured, so GCC is being built from")
    print("source. This takes 60-90 minutes. See TOOLCHAIN_GDRIVE_ID at the top")
    print("of this cell for how to do it once and never again — and for why the")
    print("Sonic 1 archive is KOS 2.2.2 / GCC 14.3.0 rather than this port's")
    print("own KOS master / 14.4.1 combination.\033[0m\n")
    n = step(f"kos-chain: binutils + GCC {TOOLCHAIN_PROFILE} + newlib (sh-elf)")
    fetch_kos()
    # utils/kos-chain on master; it was utils/dc-chain up to KOS 2.2.x, and the
    # config file was config.mk rather than Makefile.cfg. Say which it is rather
    # than failing twenty lines later on a missing sample file.
    kc = f"{DC}/kos/utils/kos-chain"
    if not os.path.isdir(kc):
        raise SystemExit(
            f"{kc} is missing.\n"
            "The toolchain builder is utils/kos-chain on KOS master and was\n"
            "utils/dc-chain up to 2.2.x, with a different config layout. If you\n"
            "moved KOS_COMMIT back to a release, this cell needs the old\n"
            "`cp config.mk.stable.sample config.mk` + `make build` flow instead.")
    prof_dir = f"{kc}/profiles/dreamcast"
    if not os.path.isfile(f"{prof_dir}/{TOOLCHAIN_PROFILE}.mk"):
        avail = ", ".join(sorted(f[:-3] for f in os.listdir(prof_dir)
                                 if f.endswith(".mk")))
        raise SystemExit(f"No kos-chain profile {TOOLCHAIN_PROFILE!r}.\n"
                         f"Available for dreamcast: {avail}")
    sh(f"cp {kc}/Makefile.dreamcast.cfg {kc}/Makefile.cfg")
    sh(f"sed -i -e 's/^toolchain_profile=.*/toolchain_profile={TOOLCHAIN_PROFILE}/' "
       f"{kc}/Makefile.cfg")
    # sh-elf only, no arm-eabi. KOS ships a precompiled AICA driver at
    # kernel/arch/dreamcast/sound/arm/stream.drv.prebuilt, and that Makefile
    # falls back to it whenever DC_ARM_CC does not exist, so sound works
    # without a second toolchain. Building one cost half an hour for nothing.
    sh("make", cwd=kc, quiet=True)
    if not have_compiler():
        raise SystemExit("kos-chain finished but sh-elf-gcc is not runnable.")
    done(n)

ver = subprocess.run([f"{DC}/sh-elf/bin/sh-elf-gcc", "--version"],
                     capture_output=True, text=True).stdout.strip().split("\n")[0]
print(f"  {ver}")

# SAY WHICH KOS IS ACTUALLY HERE, whatever route it arrived by.
#
# fetch_kos() verifies HEAD against KOS_COMMIT, but it only runs when KOS is
# cloned. A prebuilt archive brings its own KOS and skips that check entirely,
# so an archive built against a different commit would be used in silence --
# and the thing that makes this worth printing is that the development
# machine's KOS reports itself "14bc62b6-dirty". A clean checkout of the
# pinned commit is therefore NOT the same tree as the one the port is built
# against day to day. If the archive was tarred from that machine it carries
# the dirty delta; if it was tarred from a Colab kos-chain run it does not.
#
# Report, never fail: a dirty or moved KOS may well be the one that builds,
# and this cell has no way to know which. It just has to be visible in the log
# rather than discovered from a difference in behaviour later.
#
# Two details make this survive a prebuilt archive. `safe.directory`: tar run as
# root keeps the archive's numeric owners, so the KOS tree belongs to whoever
# built it (uid 1000 for the Sonic 1 archive) while Colab runs as root, and git
# refuses a repository it does not own with "detected dubious ownership" -- exit
# 128, which sh() turns into a failed build. And the try/except: this block is
# a report, so nothing in it is allowed to stop the run.
_git = f"git -c safe.directory='*' -C {DC}/kos"
if os.path.isdir(f"{DC}/kos/.git"):
    try:
        head = sh(f"{_git} rev-parse HEAD", quiet=True).strip()
        dirty = sh(f"{_git} status --porcelain", quiet=True).strip()
        desc = sh(f"{_git} describe --tags --always", quiet=True).strip()
        print(f"  KOS at {head[:8]} ({desc})" + (" -dirty" if dirty else ", clean"))
        if head != KOS_COMMIT:
            print(f"  \033[33mnote: that is NOT the pinned {KOS_COMMIT[:8]}. The ELF will")
            print("  be built against whatever this is.\033[0m")
        if dirty:
            lines = dirty.splitlines()
            print(f"  \033[33m{len(lines)} uncommitted change(s) in it:\033[0m")
            for l in lines[:10]:
                print(f"    {l}")
            if len(lines) > 10:
                print(f"    ... and {len(lines) - 10} more")
    except SystemExit as e:
        print(f"  \033[33mnote: could not identify the KOS here ({e}). Carrying on.\033[0m")
elif os.path.isdir(f"{DC}/kos"):
    print("  \033[33mnote: the KOS here is not a git checkout, so which commit it")
    print("  holds cannot be confirmed from this cell.\033[0m")

if not have_kos():
    n = step(f"fetch and build KallistiOS ({KOS_LABEL})")
    fetch_kos()
    if ENVIRON_GDRIVE_ID:
        sh("pip install -q gdown")
        sh(f"gdown -O /tmp/environ.sh {ENVIRON_GDRIVE_ID}")
        txt = open("/tmp/environ.sh", errors="ignore").read()
        # gdown hands back an HTML permission page rather than an error when a
        # file is not actually shared. Check for the real thing.
        if "KOS_BASE" not in txt:
            raise SystemExit("ENVIRON_GDRIVE_ID did not return an environ.sh — "
                             "is the file shared with anyone who has the link?")
        shutil.copyfile("/tmp/environ.sh", f"{DC}/kos/environ.sh")
    elif not os.path.isfile(f"{DC}/kos/environ.sh"):
        sh(f"cp {DC}/kos/doc/environ.sh.sample {DC}/kos/environ.sh")
    sh(f"source {DC}/kos/environ.sh && make -C {DC}/kos", quiet=True)
    if not have_kos():
        raise SystemExit("KallistiOS did not produce libkallisti.a.")
    done(n)
else:
    print("  KallistiOS already built")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  4/6  compiling the ELF\n══════════════════════════════════════════════════════════════\033[0m")
#
# The defaults in platforms/KallistiOS.cmake are the shipping configuration:
# DC_DATA_ROOT stays /cd/ so the disc build reads its data from the disc root.
# (The /pc/ dcload variant is a separate build directory; see BUILD_DREAMCAST.md.)
# RETRO_MOD_LOADER, RETRO_NETWORKING and RETRO_USE_HW_RENDER are forced off
# there, which is also deliberate -- see the comments in that file.

BUILD = f"{SRC}/build-dc"
os.makedirs(BUILD, exist_ok=True)

n = step("cmake configure")
sh(f"source {DC}/kos/environ.sh && "
   f"kos-cmake -DPLATFORM=KallistiOS -DCMAKE_BUILD_TYPE=Release {EXTRA_CMAKE_FLAGS} ..",
   cwd=BUILD)
done(n)

n = step("make -j$(nproc)")
sh(f"source {DC}/kos/environ.sh && make -j$(nproc)", cwd=BUILD, quiet=True)
done(n)

elf = None
for cand in (f"{BUILD}/RSDKv4.elf", f"{BUILD}/RetroEngine.elf"):
    if os.path.isfile(cand):
        elf = cand
        break
if not elf:
    hits = sh(f"find {BUILD} -maxdepth 2 -name '*.elf'", quiet=True).split()
    elf = hits[0] if hits else None
if not elf:
    raise SystemExit("No .elf produced — check the compile log above.")

print(f"\n{elf}   {os.path.getsize(elf) / 1e6:.2f} MB")
sh(f"source {DC}/kos/environ.sh && sh-elf-readelf -h {elf} | grep -E 'Machine|Type|Entry'")

# BSS is worth printing: the engine is almost entirely statically allocated, so
# this number is what decides how much heap is left on a 16 MB machine. Around
# 12.1 MB is normal for this port.
sh(f"source {DC}/kos/environ.sh && sh-elf-size {elf}")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  5/6  music, sound effects and menu art\n══════════════════════════════════════════════════════════════\033[0m")
#
# generate_assets.sh extracts the music, the global sfx and the 2013 menu
# textures from Data.rsdk and converts them, writing a loose Data/ tree. The
# pack is only read, never modified. The build links no Vorbis decoder, which is
# why nothing on the disc may still be a container, and the DC cannot decode a
# PNG of menu size, which is why those become .dtex. Everything else -- sprites,
# stages, scripts, palettes, and the stage sfx -- streams out of Data.rsdk
# untouched.
#
# verify_assets.sh runs at the end of generate_assets.sh and cross-checks every
# output against its source duration. That matters here more than it looks: a
# headerless file carries no sample rate, so a wrong conversion is a silent
# pitch shift rather than an error.

n = step("generate_assets.sh  (extract; music → s8 stereo 22050; global sfx → s8 mono 22050; menu → VQ dtex)")
STAGE = f"{WORK}/stage"
os.makedirs(STAGE, exist_ok=True)
sh(f"{SRC}/dreamcast/generate_assets.sh '{rsdk_path}' '{STAGE}'")
done(n)

DATA_DIR = f"{STAGE}/Data"
mus  = len([f for _, _, fs in os.walk(f"{DATA_DIR}/Music")   for f in fs])
sfx  = len([f for _, _, fs in os.walk(f"{DATA_DIR}/SoundFX") for f in fs])
menu = len([f for _, _, fs in os.walk(f"{DATA_DIR}/Game")    for f in fs])
if mus == 0 or sfx == 0:
    raise SystemExit("generate_assets.sh produced no Music or no SoundFX.")
if menu == 0:
    print("  \033[33mnote: no Data/Game came out — the 2013 menu will try to decode")
    print("  PNGs from the pack at run time and most will be invisible. That is")
    print("  what --skip-menu does, so check it was not passed.\033[0m")
print(f"\n  Music       {mus} files")
print(f"  SoundFX     {sfx} files")
print(f"  Game (menu) {menu} files")
sh(f"du -sh {DATA_DIR}/Music {DATA_DIR}/SoundFX {DATA_DIR}/Game 2>/dev/null || true")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  6/6  assembling the build kit\n══════════════════════════════════════════════════════════════\033[0m")
#
# Everything Section B needs, in one folder in this runtime. Section B reads it
# from there, so you can re-master as often as you like -- a different disc
# name, an edited settings.ini -- without recompiling anything.

n = step("write the build kit")
sh(f"rm -rf '{KIT_DIR}' && mkdir -p '{KIT_DIR}'")
shutil.copyfile(elf, f"{KIT_DIR}/RSDKv4.elf")
shutil.copyfile(rsdk_path, f"{KIT_DIR}/Data.rsdk")
sh(f"cp -R '{DATA_DIR}' '{KIT_DIR}/Data'")

# settings.ini is optional, and it is the user's choice: one uploaded beside
# Data.rsdk wins, then one committed to the repo, else none. Fabricating one
# here would be guessing at values nobody chose, so with none the disc ships
# without it and the engine uses its built-in defaults.
ini = ini_upload
for cand in (f"{SRC}/dreamcast/settings.ini", f"{SRC}/settings.ini"):
    if ini is None and os.path.isfile(cand):
        ini = cand
if ini:
    shutil.copyfile(ini, f"{KIT_DIR}/settings.ini")
    print(f"  settings.ini from {ini}")
else:
    print("  no settings.ini uploaded — the disc ships without one and the")
    print("  engine uses its built-in defaults")

def _quiet(cmd, fallback="unknown"):
    # stderr dropped: a source tree that is not a git checkout is legitimate
    # (REPO_URL may be a plain local path) and is not worth a "fatal:" line.
    try:
        return sh(f"{cmd} 2>/dev/null", quiet=True).strip() or fallback
    except SystemExit:
        return fallback

commit = _quiet(f"git -c safe.directory='*' -C {SRC} rev-parse --short HEAD")

# Record what the ELF was ACTUALLY built with, not what this cell pins. The two
# differ whenever a prebuilt archive is used -- the first full run of this
# notebook wrote "KOS master @ 14bc62b6 / GCC 14.4.0" into this file for an ELF
# that was in fact built with KOS 2.2.2 and GCC 14.3.0 from the Sonic 1 archive.
# A build record that states the pin is worse than none.
gcc_used = _quiet(f"{DC}/sh-elf/bin/sh-elf-gcc --version | head -1")
kos_used = _quiet(f"git -c safe.directory='*' -C {DC}/kos describe --tags --always --dirty")
kos_sha  = _quiet(f"git -c safe.directory='*' -C {DC}/kos rev-parse HEAD")
kos_ver  = _quiet("grep -E 'define KOS_VERSION_(MAJOR|MINOR|PATCH)' "
                  f"{DC}/kos/include/kos/version.h | awk '{{print $3}}' | paste -sd.")
with open(f"{KIT_DIR}/BUILD_INFO.txt", "w") as f:
    f.write(textwrap.dedent(f"""\
        Sonic 2 / RSDKv4 Dreamcast build kit
        built    {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}
        repo     {REPO_URL} @ {REPO_REF} ({commit})
        cmake    -DPLATFORM=KallistiOS -DCMAKE_BUILD_TYPE=Release {EXTRA_CMAKE_FLAGS}
        KOS      {kos_used}  (reports {kos_ver})
                 {kos_sha}
        GCC      {gcc_used}
        pinned   KOS {KOS_COMMIT[:8]} / GCC profile {TOOLCHAIN_PROFILE} -- used only when
                 the toolchain is built from source rather than unpacked
        """))
done(n)

sh(f"du -sh '{KIT_DIR}'/* | sort -h")
print(f"\n\033[32mBuild kit ready at {KIT_DIR}\033[0m")
print("Section A is done. Run Section B now — the kit lives in this runtime")
print("and goes with it when the session ends.")


---

## Section B — master the CDI

Press play below once Section A has finished. About two minutes.

Re-run it as often as you like — a different disc name, an edited
`settings.ini` in `/content/work/build-kit/`. It never touches the compiler.

**Same session as Section A.** The build kit lives in the runtime, not on your
Drive, so it goes when the session does.


In [ ]:
#@title ▶ Section B — master the CDI { display-mode: "form" }
#
# Press play once Section A has finished. About two minutes.
# Re-run it freely — it never touches the compiler.

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  1/4  checking the build kit\n══════════════════════════════════════════════════════════════\033[0m")

DISC_TITLE  = "SONIC2"
OUTPUT_NAME = "Sonic2.cdi"

# mkdcdisc's -N. Padding pads the data track out so the drive seeks less, which
# sounds like what a disc-bound port wants — but it inflates the image well past
# the 700 MB of a standard CD-R. Leave this ON unless you know your media can
# take it.
#
# Sonic 2's converted audio is the bulk of the image: the music alone is about
# 76 MB once it is raw 22050 stereo PCM, against 21 MB as Vorbis. Expect
# something in the region of 140 MB unpadded.
DISABLE_PADDING = True

import os, sys, time, subprocess, shutil, collections

WORK    = "/content/work"
KIT_DIR = f"{WORK}/build-kit"

_t0 = {}
def step(name):
    print(f"\n\033[1m▶ {name}\033[0m", flush=True); _t0[name] = time.time(); return name
def done(name):
    d = time.time() - _t0[name]
    print(f"\033[32m✔ {name} — {int(d // 60)}m {int(d % 60):02d}s\033[0m", flush=True)
def sh(cmd, cwd=None, quiet=False):
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = collections.deque(maxlen=300)
    for line in p.stdout:
        tail.append(line)
        if not quiet: sys.stdout.write(line)
    p.wait()
    if p.returncode:
        if quiet: print("".join(tail))
        raise SystemExit(f"command failed (exit {p.returncode}): {cmd}")
    return "".join(tail)

# settings.ini is optional — Section A says so when the repo has none — so it is
# checked for separately rather than failing the kit.
need = ["RSDKv4.elf", "Data.rsdk", "Data"]
missing = [x for x in need if not os.path.exists(f"{KIT_DIR}/{x}")]
if missing:
    raise SystemExit(f"build kit at {KIT_DIR} is missing: {', '.join(missing)}\n"
                     f"Run Section A first, in this same session.")
HAVE_INI = os.path.isfile(f"{KIT_DIR}/settings.ini")
HAVE_MENU = os.path.isdir(f"{KIT_DIR}/Data/Game")

if os.path.isfile(f"{KIT_DIR}/BUILD_INFO.txt"):
    print(open(f"{KIT_DIR}/BUILD_INFO.txt").read())
sh(f"du -sh '{KIT_DIR}'/* | sort -h")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  2/4  building mkdcdisc\n══════════════════════════════════════════════════════════════\033[0m")
#
# Stock Ubuntu packages, no vendored subprojects, ~20 seconds on 2 vCPUs.
#
# -Dpng=enabled rather than the default: with 'auto', a missing libpng silently
# produces a build without PNG support instead of an error. Convenient locally,
# wrong in a pipeline.

MKDCDISC_REF = "main"

n = step("apt packages for mkdcdisc")
sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "meson ninja-build pkg-config libisofs-dev libpng-dev build-essential > /dev/null",
   quiet=True)
done(n)

MK = f"{WORK}/mkdcdisc"
if not os.path.isfile(f"{MK}/builddir/mkdcdisc"):
    n = step("clone and build mkdcdisc")
    if not os.path.isdir(MK):
        sh(f"git clone --depth 1 --branch {MKDCDISC_REF} "
           f"https://gitlab.com/simulant/mkdcdisc.git {MK}")
    sh("meson setup builddir -Dpng=enabled", cwd=MK)
    sh("ninja -C builddir", cwd=MK, quiet=True)
    done(n)

# --help, not --version: mkdcdisc has no --version and exits 2 on it, which
# prints its banner followed by an error and looks like a broken build.
sh(f"{MK}/builddir/mkdcdisc --help | head -2")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  3/4  mastering\n══════════════════════════════════════════════════════════════\033[0m")
#
# The converted Data/ tree goes on the disc NEXT TO the untouched Data.rsdk.
# The engine reads /cd/Data.rsdk for everything except music, global sfx and the
# 2013 menu art, which it opens loose from /cd/Data/ — so a disc with only one
# of the two is a disc that boots and then plays silent with half a menu.

n = step("stage the kit")
STAGE = f"{WORK}/master"
sh(f"rm -rf {STAGE} && mkdir -p {STAGE}")
for item in ("RSDKv4.elf", "Data.rsdk"):
    shutil.copyfile(f"{KIT_DIR}/{item}", f"{STAGE}/{item}")
if HAVE_INI:
    shutil.copyfile(f"{KIT_DIR}/settings.ini", f"{STAGE}/settings.ini")
sh(f"cp -R '{KIT_DIR}/Data' '{STAGE}/Data'")
done(n)

n = step("mkdcdisc")
OUT = f"/content/{OUTPUT_NAME}"
# mkdcdisc refuses to overwrite an existing image and exits 4. Re-mastering is
# the whole point of Section B being separate, so the second run must not be the
# one that fails: clear the target rather than relying on a flag.
if os.path.exists(OUT):
    os.remove(OUT)
pad = "-N " if DISABLE_PADDING else ""
ini = f"-f '{STAGE}/settings.ini' " if HAVE_INI else ""
cmd = (f"{MK}/builddir/mkdcdisc "
       f"-e '{STAGE}/RSDKv4.elf' "
       f"-d '{STAGE}/Data' "
       f"-f '{STAGE}/Data.rsdk' "
       f"{ini}"
       f"{pad}-n {DISC_TITLE} -o '{OUT}'")
print(cmd + "\n")
sh(cmd)
done(n)

print(f"\n\033[32m{OUT}   {os.path.getsize(OUT) / 1e6:.0f} MB\033[0m")

print("\n\033[1;36m══════════════════════════════════════════════════════════════\n  4/4  verifying\n══════════════════════════════════════════════════════════════\033[0m")
#
# Not a substitute for booting it, but it catches the failures that are silent
# otherwise: a disc with no boot signature, or one that mastered "successfully"
# while quietly dropping a directory.

with open(OUT, "rb") as f:
    blob = f.read()

checks = [
    (b"SEGA SEGAKATANA", "Dreamcast boot signature"),
    (b"1ST_READ.BIN",    "scrambled executable"),
    (b"DATA.RSDK",       "the datafile"),
    (b"MUSIC",           "Data/Music"),
    (b"SOUNDFX",         "Data/SoundFX"),
    (b"GLOBAL",          "Data/SoundFX/Global"),
]
if HAVE_MENU:
    # The 2013 menu art is a third loose class on this port, and a dropped
    # Game/ directory is exactly the kind of silent miss this section exists
    # for -- the disc boots and the menu is simply missing pieces.
    checks.append((b"MENU", "Data/Game/Menu"))
if HAVE_INI:
    checks.append((b"SETTINGS.INI", "settings"))
bad = 0
for needle, label in checks:
    ok = needle in blob
    bad += not ok
    print(f"  {'✔' if ok else '✘'}  {label}")
del blob

print(f"\n  {os.path.getsize(OUT) / 1e6:.0f} MB")
if bad:
    print("\n\033[31mSomething is missing from the image — do not burn this.\033[0m")
else:
    print("\n\033[32mImage looks structurally sound.\033[0m")
    print(f"\nDownload it from the \033[1mFiles\033[0m tab in the left sidebar:")
    print(f"    {OUT}")
    print("\nRight-click the file → Download. Use the sidebar rather than the")
    print("line below: a few hundred MB through the notebook connection is a")
    print("fragile transfer, and the sidebar is the more reliable route.")
    print("\nIf you would rather have it start automatically, run:")
    print("    from google.colab import files; files.download(OUT)")
